In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 77 (delta 21), reused 17 (delta 17), pack-reused 48 (from 1)
Receiving objects: 100% (77/77), 1.05 MiB | 4.14 MiB/s, done.
Resolving deltas: 100% (35/35), done.


In [3]:
pip install ucimlrepo

In [4]:
from ucimlrepo import fetch_ucirepo

In [6]:
pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 8.2 MB/s eta 0:00:00


In [7]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
from sdv.metadata import SingleTableMetadata
import torch
import random

concrete = fetch_ucirepo(id=165)
X = concrete.data.features
y = concrete.data.targets
print(concrete.metadata)
print(concrete.variables)

data = pd.concat([X, y], axis=1)
target_col = "Concrete compressive strength"

# Drop date/time/session/ID features before generator training (high cardinality).
_drop_feature_cols = [
    "Date", "Time", "date_time",
    "session_id", "Session ID", "Session_ID", "session",
    "X1 transaction date",
]
data = data.drop(columns=[c for c in _drop_feature_cols if c in data.columns], errors="ignore")

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data[col] = data[col].fillna(data[col].median())

processed_data = data.copy()

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


{'uci_id': 165, 'name': 'Concrete Compressive Strength', 'repository_url': 'https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength', 'data_url': 'https://archive.ics.uci.edu/static/public/165/data.csv', 'abstract': 'Concrete is the most important material in civil engineering. The concrete compressive strength is a highly nonlinear function of age and ingredients. ', 'area': 'Physics and Chemistry', 'tasks': ['Regression'], 'characteristics': ['Multivariate'], 'num_instances': 1030, 'num_features': 8, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Concrete compressive strength'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1998, 'last_updated': 'Sun Feb 11 2024', 'dataset_doi': '10.24432/C5PK67', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 383, 'type': 'NATIVE', 'title': 'Modeling of strength of high-performance concrete using artificial neural networks', 'authors': 'I. Yeh', 'venue': 'C

In [8]:
from sklearn.model_selection import train_test_split
from sdv.evaluation.single_table import evaluate_quality

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
        random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

try:
    data_path = "concrete_train.csv"
    train_real.to_csv(data_path, index=False)

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=[],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Regression": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    # Convert all columns back to numeric where possible
    for col in synthetic_ctabgan.columns:
        synthetic_ctabgan[col] = pd.to_numeric(
            synthetic_ctabgan[col],
            errors="coerce"
        )

        synthetic_ctabgan[col] = synthetic_ctabgan[col].fillna(
            train_real[col].median()
        )

    # Concrete Compressive Strength quality target is an integer score between 0 and 10
    pass  # keep continuous regression target as-is

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)



================ SINGLE RUN ================


100%|██████████| 150/150 [00:53<00:00,  2.79it/s]


Finished training in 62.90093946456909  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 282.53it/s]|
Column Shapes Score: 76.79%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 243.15it/s]|
Column Pair Trends Score: 73.34%

Overall Score (Average): 75.06%

CTABGAN: 0.7506


In [10]:
import traceback
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim

# WGAN-GP

try:

    data_wgan = train_real.copy()

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    # max_label_code = len(encoder.classes_) - 1 # This line is commented out as 'encoder' is not defined and not needed for this numerical dataset.

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 268.38it/s]|
Column Shapes Score: 80.15%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 224.61it/s]|
Column Pair Trends Score: 97.93%

Overall Score (Average): 89.04%

WGAN_GP: 0.8904


In [11]:
# SDV MODELS

from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        pass  # keep continuous regression target as-is

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 258.37it/s]|
Column Shapes Score: 74.71%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 245.62it/s]|
Column Pair Trends Score: 69.02%

Overall Score (Average): 71.87%

CTGAN: 0.7187
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 213.92it/s]|
Column Shapes Score: 68.58%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 248.70it/s]|
Column Pair Trends Score: 72.29%

Overall Score (Average): 70.43%

CopulaGAN: 0.7043
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 335.52it/s]|
Column Shapes Score: 85.97%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 243.12it/s]|
Column Pair Trends Score: 96.56%

Overall Score (Average): 91.27%

TVAE: 0.9127
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 292.36it/s]|
Column Shapes Score: 7

In [18]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

EVAL_SEEDS = [42, 43, 44, 45, 46] # Changed to multiple seeds for robust evaluation
GENERATORS_TO_EVAL = list(synthetic_datasets.keys())

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')


Regression evaluation: 10 models, 5 seeds, 6 generators


In [19]:
def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    train_df = align_to_train_schema(train_df, schema_df, label_col)
    test_df = align_to_train_schema(test_df, schema_df, label_col)
    results = []

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if not use_holdout:
                X_train, _, y_train, _ = train_test_split(
                    X_train, y_train, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': np.std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': np.std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': np.std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': np.std(mae_scores),
            'R2 (Mean±Std)': f"{np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}",
            'MSE (Mean±Std)': f"{np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}",
            'RMSE (Mean±Std)': f"{np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}",
            'MAE (Mean±Std)': f"{np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [21]:
def align_to_train_schema(df_to_align, schema_reference_df, label_col):
    # Identify feature columns from the schema reference
    feature_cols_schema = [col for col in schema_reference_df.columns if col != label_col]

    # Handle feature columns first
    for col in feature_cols_schema:
        if col not in df_to_align.columns:
            # Add missing feature columns, fill with the median from the schema_reference_df
            df_to_align[col] = schema_reference_df[col].median()

    # Drop any extra columns in df_to_align that are not in schema_reference_df
    extra_cols = [col for col in df_to_align.columns if col not in schema_reference_df.columns]
    if extra_cols:
        df_to_align = df_to_align.drop(columns=extra_cols)

    # Reorder columns of df_to_align to match schema_reference_df
    final_ordered_cols = [col for col in schema_reference_df.columns if col in df_to_align.columns]
    df_to_align = df_to_align[final_ordered_cols]

    return df_to_align

print('TRTR (Train Real, Test Real) — 80% train / 20% holdout')
trtr_results = evaluate_regression_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=False, # Changed to False to introduce variability in splits
    schema_df=train_real,
)
display(trtr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on 20% holdout)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=False, # Changed to False to introduce variability in splits
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


TRTR (Train Real, Test Real) — 80% train / 20% holdout


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
7,RandomForest,0.9029 ± 0.0106,24.0448 ± 5.7102,4.8678 ± 0.5913,3.4008 ± 0.4083
8,ExtraTrees,0.9025 ± 0.0142,24.2633 ± 6.8688,4.8775 ± 0.6878,3.2496 ± 0.4946
9,GradientBoost,0.8902 ± 0.0141,27.2738 ± 7.3724,5.1763 ± 0.6926,3.8355 ± 0.3801
6,DecisionTree,0.7582 ± 0.0804,59.5759 ± 22.9523,7.5790 ± 1.4608,4.9940 ± 0.7970
5,KNN,0.6311 ± 0.0815,88.8531 ± 18.5112,9.3772 ± 0.9594,7.5429 ± 0.8544
2,Lasso,0.5756 ± 0.0588,102.9093 ± 16.2255,10.1129 ± 0.7993,7.8022 ± 0.7718
3,ElasticNet,0.5756 ± 0.0588,102.9094 ± 16.2257,10.1129 ± 0.7993,7.8022 ± 0.7718
1,Ridge,0.5756 ± 0.0588,102.9094 ± 16.2258,10.1129 ± 0.7993,7.8022 ± 0.7718
0,LinearRegression,0.5756 ± 0.0588,102.9096 ± 16.2263,10.1129 ± 0.7993,7.8022 ± 0.7718
4,SVR_RBF,0.2168 ± 0.0345,191.6581 ± 29.7800,13.8000 ± 1.1036,11.0860 ± 1.1607


CTABGAN - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
0,LinearRegression,0.0813 ± 0.0902,225.0178 ± 38.8779,14.9369 ± 1.3806,12.3253 ± 1.3545
1,Ridge,0.0813 ± 0.0902,225.0182 ± 38.8778,14.9369 ± 1.3806,12.3253 ± 1.3545
3,ElasticNet,0.0813 ± 0.0902,225.0189 ± 38.8777,14.9370 ± 1.3806,12.3253 ± 1.3545
2,Lasso,0.0813 ± 0.0902,225.0197 ± 38.8775,14.9370 ± 1.3806,12.3253 ± 1.3545
8,ExtraTrees,0.0762 ± 0.1143,227.6068 ± 47.0990,14.9931 ± 1.6772,12.2841 ± 1.6642
7,RandomForest,0.0298 ± 0.1478,238.7297 ± 53.2313,15.3463 ± 1.7948,12.5910 ± 1.8637
4,SVR_RBF,-0.0900 ± 0.0807,266.5547 ± 41.9922,16.2712 ± 1.3422,13.1539 ± 1.5656
5,KNN,-0.1454 ± 0.1794,282.0024 ± 72.3704,16.6604 ± 2.1056,13.5611 ± 2.2097
9,GradientBoost,-0.2177 ± 0.1993,291.7670 ± 27.3295,17.0624 ± 0.8014,14.2462 ± 0.8752
6,DecisionTree,-1.0040 ± 0.6227,484.2053 ± 161.1664,21.7143 ± 3.5627,17.7801 ± 3.0711


WGAN_GP - TSTR (train on synthetic, test on 20% holdout)


...

TVAE - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
7,RandomForest,0.4934 ± 0.1003,123.8922 ± 30.2854,11.0448 ± 1.3800,8.6473 ± 1.0474
8,ExtraTrees,0.4807 ± 0.0872,125.2806 ± 19.6344,11.1582 ± 0.8808,8.7000 ± 0.7399
1,Ridge,0.4593 ± 0.0627,132.9624 ± 27.7325,11.4641 ± 1.2392,9.0385 ± 0.9353
3,ElasticNet,0.4593 ± 0.0627,132.9631 ± 27.7329,11.4642 ± 1.2393,9.0385 ± 0.9353
2,Lasso,0.4593 ± 0.0627,132.9631 ± 27.7329,11.4642 ± 1.2393,9.0385 ± 0.9353
0,LinearRegression,0.4592 ± 0.0627,132.9657 ± 27.7344,11.4643 ± 1.2393,9.0386 ± 0.9353
9,GradientBoost,0.3846 ± 0.1524,148.4638 ± 36.9337,12.0874 ± 1.5362,9.3213 ± 1.3246
4,SVR_RBF,0.2971 ± 0.0536,171.3180 ± 24.8213,13.0537 ± 0.9580,10.7101 ± 1.0346
5,KNN,0.2688 ± 0.1793,174.4208 ± 29.7070,13.1546 ± 1.1732,10.5928 ± 0.8991
6,DecisionTree,-0.1759 ± 0.3025,291.1012 ± 91.6442,16.8353 ± 2.7700,13.2650 ± 2.0243


GaussianCopula - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean±Std),MSE (Mean±Std),RMSE (Mean±Std),MAE (Mean±Std)
8,ExtraTrees,0.7117 ± 0.0634,72.2529 ± 22.9053,8.3802 ± 1.4232,6.7417 ± 1.1295
7,RandomForest,0.6960 ± 0.0517,75.5280 ± 20.7266,8.6023 ± 1.2362,6.7695 ± 0.9404
9,GradientBoost,0.5612 ± 0.0909,107.7699 ± 29.2831,10.2902 ± 1.3717,8.3945 ± 1.0997
1,Ridge,0.4933 ± 0.1126,123.4109 ± 32.1020,11.0211 ± 1.3948,8.8452 ± 1.2348
3,ElasticNet,0.4933 ± 0.1126,123.4218 ± 32.1068,11.0216 ± 1.3950,8.8457 ± 1.2349
2,Lasso,0.4932 ± 0.1127,123.4255 ± 32.1081,11.0218 ± 1.3950,8.8458 ± 1.2349
0,LinearRegression,0.4932 ± 0.1127,123.4476 ± 32.1197,11.0227 ± 1.3953,8.8468 ± 1.2352
5,KNN,0.3886 ± 0.1590,147.8239 ± 39.9531,12.0520 ± 1.6043,9.6978 ± 1.4379
4,SVR_RBF,0.3119 ± 0.0484,168.8711 ± 30.8660,12.9393 ± 1.2019,10.3694 ± 1.1531
6,DecisionTree,0.0270 ± 0.2505,237.8010 ± 70.3665,15.2450 ± 2.3221,12.2943 ± 2.1442


,Synthetic_Model,R2_Drop,MSE_Increase,RMSE_Increase,MAE_Increase
3,GaussianCopula,0.193480,47.644576,2.546678,2.433312
4,TVAE,0.301847,73.902422,3.706137,3.207297
5,WGAN_GP,0.331507,80.107369,3.967915,3.251137
0,CTABGAN,0.763002,186.363371,7.566612,6.760000
1,CTGAN,1.235679,293.885119,10.297431,8.753399
2,CopulaGAN,1.512751,370.762893,12.528827,10.385845


In [23]:
output_file = 'TRTR_TSTR_results_air_quality.xlsx'

# Create quality_df from the scores dictionary
quality_df = pd.DataFrame.from_dict(scores, orient='index', columns=['Quality Score'])
quality_df.index.name = 'Synthetic Model'
quality_df = quality_df.reset_index()

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')


Results saved to: TRTR_TSTR_results_air_quality.xlsx
